# 01 — Analisi del Portfolio (Fase 1)

Questo notebook mostra l'intero flusso della **Fase 1** di `trading-ai`:

1. Caricamento dei dati (qui usiamo dati di esempio che imitano il formato reale dei broker)
2. Calcolo di tutte le metriche di portfolio (P&L FIFO, rendimenti, rischio, allocazione)
3. Visualizzazione interattiva con Plotly

> **Nota:** i prezzi sono *iniettati* da uno storico sintetico per garantire riproducibilità offline.
> Con i CSV reali in `data/raw/`, basta usare `loader.load_transactions()` e rimuovere gli override `price_data` / `current_prices` per scaricare i prezzi veri da yfinance.

## 0. Setup
Aggiungiamo la root del progetto al `sys.path` così possiamo importare il package `src` senza installarlo, e abilitiamo il rendering Plotly inline.

In [1]:
import sys
from pathlib import Path

# Root = cartella che contiene `src/` (un livello sopra `notebooks/`).
ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = 'iframe'

import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Root progetto:', ROOT)

Root progetto: /Users/lorenzogennari/Desktop/agent ai/trading-ai


## 1. Caricamento dati
Generiamo transazioni di esempio nello **schema canonico** (`date, ticker, type, quantity, price, amount_eur, currency, fx_rate, broker`).
Il dataset include acquisti multi-lotto, una vendita parziale, una posizione in USD, dividendi, commissioni e imposte.

Con dati reali sostituire con:
```python
from src.ingestion import loader
tx = loader.load_transactions()      # legge tutti i CSV in data/raw/
loader.save_processed(tx)            # salva in data/processed/transactions.parquet
```

In [2]:
from src.ingestion.sample_data import make_sample_transactions, make_sample_price_history

tx = make_sample_transactions()
price_history = make_sample_price_history(tx)
# Prezzi correnti fittizi (EUR) per l'unrealized; con dati reali ometterli per usare yfinance.
current_prices = {'VWCE.DE': 130.0, 'AAPL': 200.0, 'ENI.MI': 15.0}

tx

,date,ticker,type,quantity,price,amount_eur,currency,fx_rate,broker
0,2023-01-10,VWCE.DE,buy,50.00,100.00,"-5,000.00",EUR,1.00,sample
1,2023-01-10,CASH,fee,NaN,NaN,-7.50,EUR,1.00,sample
2,2023-02-01,AAPL,buy,20.00,150.00,"-2,760.00",USD,0.92,sample
3,2023-03-05,ENI.MI,buy,100.00,13.50,"-1,350.00",EUR,1.00,sample
4,2023-04-15,VWCE.DE,buy,30.00,110.00,"-3,300.00",EUR,1.00,sample
5,2023-05-20,ENI.MI,dividend,0.00,0.00,45.00,EUR,1.00,sample
6,2023-06-01,AAPL,buy,10.00,180.00,"-1,674.00",USD,0.93,sample
7,2023-08-15,AAPL,dividend,0.00,0.00,22.75,USD,0.91,sample
8,2023-09-20,VWCE.DE,sell,40.00,120.00,"4,800.00",EUR,1.00,sample
9,2023-09-20,CASH,tax,NaN,NaN,-32.00,EUR,1.00,sample


## 2. Metriche di portfolio

### 2.1 Posizioni attuali e P&L realizzato (FIFO)
Il P&L realizzato abbina le vendite ai lotti di acquisto più vecchi (FIFO), gestendo le vendite parziali.

In [3]:
from src.analysis import metrics

print('Posizioni attuali (quantità):')
display(metrics.current_holdings(tx).to_frame())

print('\nP&L realizzato (FIFO) per ticker:')
realized = metrics.realized_pnl(tx)
display(realized)

Posizioni attuali (quantità):


,quantity
ticker,
AAPL,30.00
ENI.MI,100.00
VWCE.DE,40.00



P&L realizzato (FIFO) per ticker:


,realized_pnl_eur
ticker,
VWCE.DE,800.00


### 2.2 P&L non realizzato (mark-to-market)
Confronta il costo dei lotti residui col valore di mercato attuale.

In [4]:
unrealized = metrics.unrealized_pnl(tx, current_prices=current_prices)
display(unrealized)

,quantity,avg_cost_eur,cost_basis_eur,current_price,market_value_eur,unrealized_pnl_eur
ticker,,,,,,
AAPL,30.00,147.80,"4,434.00",200.00,"6,000.00","1,566.00"
VWCE.DE,40.00,107.50,"4,300.00",130.00,"5,200.00",900.00
ENI.MI,100.00,13.50,"1,350.00",15.00,"1,500.00",150.00


### 2.3 Valore del portfolio nel tempo
Posizioni cumulate giornaliere valorizzate con lo storico prezzi, più la componente di cassa.

In [5]:
pv = metrics.portfolio_value_over_time(tx, price_data=price_history)
pv.tail()

,holdings_value,cash,total_value
date,,,
2026-06-19,"10,403.31","-9,255.75","1,147.56"
2026-06-20,"10,518.72","-9,255.75","1,262.97"
2026-06-21,"10,545.85","-9,255.75","1,290.10"
2026-06-22,"10,508.04","-9,255.75","1,252.29"
2026-06-23,"10,464.26","-9,255.75","1,208.51"


### 2.4 Rendimenti (TWR / MWR) e metriche di rischio

In [6]:
ret = metrics.total_return(tx, price_data=price_history, current_prices=current_prices)
# Le metriche di rischio usano holdings_value (valore di mercato delle posizioni):
# total_value include la cassa che, senza i versamenti tracciati, puo' essere negativa.
mdd = metrics.max_drawdown(pv['holdings_value'])
sharpe = metrics.sharpe_ratio(pv['holdings_value'])

print(f"TWR (time-weighted):  {ret['twr']:.2%}")
print(f"MWR / XIRR (annuo):   {ret['mwr']:.2%}")
print(f"Valore finale:        EUR {ret['final_value']:,.2f}")
print(f"Max Drawdown:         {mdd:.2%}")
print(f"Sharpe ratio:         {sharpe:.2f}")

TWR (time-weighted):  13.12%
MWR / XIRR (annuo):   9.18%
Valore finale:        EUR 12,700.00
Max Drawdown:         -52.67%
Sharpe ratio:         0.48


### 2.5 Allocazione attuale

In [7]:
alloc = metrics.allocation_breakdown(tx, current_prices=current_prices)
display(alloc)
print('\nPer asset class:')
display(alloc.groupby('asset_class')['weight_pct'].sum().sort_values(ascending=False).to_frame())

,market_value_eur,asset_class,weight_pct
ticker,,,
AAPL,"6,000.00",Equity,47.24
VWCE.DE,"5,200.00",ETF,40.94
ENI.MI,"1,500.00",ETF,11.81



Per asset class:


,weight_pct
asset_class,
ETF,52.76
Equity,47.24


## 3. Visualizzazioni (Plotly)

### 3.1 Valore del portfolio con shading del drawdown

In [8]:
from src.analysis import charts

charts.plot_portfolio_value(pv).show()

### 3.2 Allocazione (torta interattiva)

In [9]:
charts.plot_allocation_pie(alloc).show()

### 3.3 Waterfall del P&L realizzato

In [10]:
charts.plot_pnl_waterfall(realized).show()

### 3.4 Heatmap dei rendimenti mensili

In [11]:
charts.plot_monthly_returns_heatmap(pv).show()

---
## Prossimi passi
- **Fase 2** — `src/backtest/`: motore di backtest delle strategie
- **Fase 3** — `src/ml/`: modelli predittivi
- **Fase 4** — `src/agent/` + `dashboard/`: agente AI (Claude) e dashboard Streamlit

Le funzioni di `src/analysis/metrics.py` sono già pensate per essere chiamate direttamente come *tool* dell'agente della Fase 4.